# 03 - Explore the transport-line reference

**Source:** `referentiel-des-lignes.csv` (full file).
**Module:** `scripts/datasets/lines.py`.

Inspect line identifiers, modes, operators and statuses separately. A reference may
include active and planned services. Its presence does not establish whether a line
operated at a particular date or time.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open the hackathon project folder before running this notebook.')
if not Path(sys.executable).resolve().is_relative_to(ROOT / '.venv'):
    raise RuntimeError('Select the project .venv kernel, then restart and run all.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.datasets import lines
from scripts.exploration import column_summary, repeated_keys, save_parquet

pio.renderers.default = 'plotly_mimetype'
print('Python:', sys.executable)


Python: D:\Hackathon Mobilités 2026 Spécial Dataviz\.venv\Scripts\python.exe


In [2]:
source = ROOT / lines.FILENAME
raw = lines.load(source)


In [3]:
print('Loaded rows and columns:', raw.shape)
display(raw.head())
raw.info()


Loaded rows and columns: (2131, 28)


,ID_Line,Name_Line,ShortName_Line,TransportMode,TransportSubmode,Type,OperatorRef,OperatorName,AdditionalOperatorsRef,NetworkName,...,ShortName_GroupOfLines,Notice_Title,Notice_Text,Picto,Valid_fromDate,Valid_toDate,Status,PrivateCode,Air_Conditioning,ID_Bus_Contrat
0,C00001,482,482,bus,regionalBus,,1060,Keolis Grand Paris Seine Orly,,Seine Orly,...,VILLENEUVE-LE-ROI (place Soupault) - VILLENEU...,,,,2014-07-16,,active,522522482,partial,47
1,C00004,483,483,bus,regionalBus,,1060,Keolis Grand Paris Seine Orly,,Seine Orly,...,CHOISY-LE-ROI (Gare RER) - VILLENEUVE-ST-GEOR...,,,,2014-07-16,,active,522522483,partial,47
2,C00007,282,282,bus,localBus,,1060,Keolis Grand Paris Seine Orly,,Seine Orly,...,THIAIS (E. LEVASSEUR) - VITRY-SUR-SEINE (Cent...,,,,2014-07-16,,active,522522282,partial,47
3,C00011,3105,3105,bus,localBus,,1025,Keolis Portes et Val de Brie,,Pays Briard,...,EVRY-GREGY-SUR-YERRE - OZOIR-LA-FERRIERE,,,,2014-07-16,,active,513513005,partial,13
4,C00012,3107,3107,bus,regionalBus,,1025,Keolis Portes et Val de Brie,,Pays Briard,...,TOURNAN-EN-BRIE - TOURNAN-EN-BRIE,,,,2014-07-16,,active,513513107,partial,13


<class 'pandas.DataFrame'>
RangeIndex: 2131 entries, 0 to 2130
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   ID_Line                 2131 non-null   string
 1   Name_Line               2131 non-null   string
 2   ShortName_Line          2131 non-null   string
 3   TransportMode           2131 non-null   string
 4   TransportSubmode        2131 non-null   string
 5   Type                    2131 non-null   string
 6   OperatorRef             2131 non-null   string
 7   OperatorName            2131 non-null   string
 8   AdditionalOperatorsRef  2131 non-null   string
 9   NetworkName             2131 non-null   string
 10  ColourWeb_hexa          2131 non-null   string
 11  TextColourWeb_hexa      2131 non-null   string
 12  ColourPrint_CMJN        2131 non-null   string
 13  TextColourPrint_hexa    2131 non-null   string
 14  Accessibility           2131 non-null   string
 15  Audiblesigns_Av

## Clean a copy and inspect the columns

The original file and `raw` table stay unchanged. Strip surrounding spaces and turn
blank cells into missing values; preserve special codes such as `NA` and `ZZ`.
Dataset-specific numeric conversions are documented in the module's `clean()` function.
Unexpected numeric text raises an error for inspection. No rows are dropped or deduplicated.


In [ ]:
cleaned = lines.clean(raw)
assert len(cleaned) == len(raw)
display(cleaned.head())
display(column_summary(cleaned))

,ID_Line,Name_Line,ShortName_Line,TransportMode,TransportSubmode,Type,OperatorRef,OperatorName,AdditionalOperatorsRef,NetworkName,...,ShortName_GroupOfLines,Notice_Title,Notice_Text,Picto,Valid_fromDate,Valid_toDate,Status,PrivateCode,Air_Conditioning,ID_Bus_Contrat
0,C00001,482,482,bus,regionalBus,<NA>,1060,Keolis Grand Paris Seine Orly,<NA>,Seine Orly,...,VILLENEUVE-LE-ROI (place Soupault) - VILLENEU...,<NA>,<NA>,<NA>,2014-07-16,<NA>,active,522522482,partial,47
1,C00004,483,483,bus,regionalBus,<NA>,1060,Keolis Grand Paris Seine Orly,<NA>,Seine Orly,...,CHOISY-LE-ROI (Gare RER) - VILLENEUVE-ST-GEOR...,<NA>,<NA>,<NA>,2014-07-16,<NA>,active,522522483,partial,47
2,C00007,282,282,bus,localBus,<NA>,1060,Keolis Grand Paris Seine Orly,<NA>,Seine Orly,...,THIAIS (E. LEVASSEUR) - VITRY-SUR-SEINE (Cent...,<NA>,<NA>,<NA>,2014-07-16,<NA>,active,522522282,partial,47
3,C00011,3105,3105,bus,localBus,<NA>,1025,Keolis Portes et Val de Brie,<NA>,Pays Briard,...,EVRY-GREGY-SUR-YERRE - OZOIR-LA-FERRIERE,<NA>,<NA>,<NA>,2014-07-16,<NA>,active,513513005,partial,13
4,C00012,3107,3107,bus,regionalBus,<NA>,1025,Keolis Portes et Val de Brie,<NA>,Pays Briard,...,TOURNAN-EN-BRIE - TOURNAN-EN-BRIE,<NA>,<NA>,<NA>,2014-07-16,<NA>,active,513513107,partial,13


,data_type,missing_rows,distinct_values
column,,,
ID_Line,string,0,2131
Name_Line,string,0,2089
ShortName_Line,string,0,1916
TransportMode,string,0,6
TransportSubmode,string,563,11
Type,string,2029,2
OperatorRef,string,0,63
OperatorName,string,0,63
AdditionalOperatorsRef,string,2100,3


## Analyse reference contents

Only whitespace/blanks are cleaned. Statuses, boolean-like values and date strings stay
as supplied so you can inspect their conventions before choosing conversions.


In [5]:
line_modes = lines.mode_summary(cleaned)
display(line_modes)
display(cleaned['Status'].value_counts(dropna=False).rename('reference_records'))
display(lines.operator_summary(cleaned).head(15))
display(repeated_keys(cleaned, ['ID_Line']))


,TransportMode,records,distinct_line_ids
0,bus,2069,2069
1,cableway,1,1
2,funicular,1,1
3,metro,18,18
4,rail,27,27
5,tram,15,15


Status
active                  2122
prochainement active       9
Name: reference_records, dtype: int64[pyarrow]

,OperatorName,reference_records
56,Transdev Sud Yvelines,109
20,Keolis Seine et Oise Est,89
45,Transdev Brie et 2 Morin,72
47,Transdev Cœur Essonne,66
39,RD Mantois,65
19,Keolis Roissy Pays de France Ouest,65
15,Keolis Grand Paris Vallée de la Marne,62
49,Transdev Marne-et-Ourcq,62
17,Keolis Portes et Val de Brie,56
8,Francilité Saint-Quentin-en-Yvelines,55


,ID_Line,records


In [6]:
lines.plot_modes(line_modes).show()


## Play with one mode

Change the value after checking the actual mode codes above. `ID_Line` remains text.
Before future joins, check identifier correspondence and source periods explicitly.
The mode-summary table is the text alternative to the chart.


In [7]:
MODE = 'bus'
display(cleaned.loc[cleaned.TransportMode.eq(MODE),
    ['ID_Line', 'Name_Line', 'TransportMode', 'OperatorName', 'Status', 'Valid_fromDate', 'Valid_toDate']].head(20))


,ID_Line,Name_Line,TransportMode,OperatorName,Status,Valid_fromDate,Valid_toDate
0,C00001,482,bus,Keolis Grand Paris Seine Orly,active,2014-07-16,<NA>
1,C00004,483,bus,Keolis Grand Paris Seine Orly,active,2014-07-16,<NA>
2,C00007,282,bus,Keolis Grand Paris Seine Orly,active,2014-07-16,<NA>
3,C00011,3105,bus,Keolis Portes et Val de Brie,active,2014-07-16,<NA>
4,C00012,3107,bus,Keolis Portes et Val de Brie,active,2014-07-16,<NA>
5,C00017,3118,bus,Keolis Portes et Val de Brie,active,2014-07-16,<NA>
6,C00021,3145,bus,Keolis Portes et Val de Brie,active,2014-07-16,<NA>
7,C00030,770,bus,Keolis Grand Paris Vallée de la Marne,active,2014-07-16,<NA>
8,C00032,760,bus,Keolis Grand Paris Vallée de la Marne,active,2014-07-16,<NA>
9,C00038,6132,bus,Keolis Vélizy Vallée de la Bièvre,active,2014-07-16,<NA>


## Optional: save this dataset as Parquet



In [8]:
SAVE_CLEANED = False
output_path = ROOT / 'data' / 'cleaned' / 'lines.parquet'

if SAVE_CLEANED:
    save_parquet(cleaned, output_path)
    print('Saved', len(cleaned), 'rows to', output_path)
else:
    print('No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.')


No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.
